In [ ]:
import requests

#PORT = 8008
#SERVER = "http://127.0.0.1"
PORT = 80
SERVER = "http://192.168.2.239"

podcasts = requests.get(f"{SERVER}:{PORT}/api/podcasts/")
podcasts = podcasts.json()

In [ ]:
# get the channel slug, episode guid, and transcription uuid for all transcriptions without a spaCy segmentation
transcripts = []
for podcast in podcasts:
    for audioitem in podcast['audioitem_set']:
        for transcription in audioitem['transcription_set']:
            # check if any of the objects in segmentation_set have name == "spaCy"
            if not any(segmentation['name'] == "spaCy" for segmentation in transcription['segmentation_set']):
                transcripts.append({"slug": podcast['slug'], "ep_guid": audioitem['guid'], "trans_uuid": transcription['uuid']})
len(transcripts)

In [ ]:
failed = []

In [ ]:
import sentence_splitter
import spacy
spacy_dict = {"en": "en_core_web_lg", "no": "nb_core_news_lg", "nb": "nb_core_news_lg", "de": "de_dep_news_trf", "sv": "sv_core_news_lg", "da": "da_core_news_trf"}

for transcript in transcripts:
    slug = transcript["slug"]
    ep_guid = transcript["ep_guid"]
    trans_uuid = transcript["trans_uuid"]

    # Get the text from the API
    data = requests.get(f"{SERVER}:{PORT}/api/transcriptions/{trans_uuid}/")
    transcript = data.json()
    
    lang = transcript["item"]["channel"]["language"][0:2]
    try:
        # Split the text into sentences
        utterances = sentence_splitter.sentence_splitter(transcript, spacy_dict[lang])

        segmentation_dict = {
            "uuid": trans_uuid,
            "name": "spaCy",
            "segmentor": {"name": "spaCy", "version": spacy.__version__},
            "utterance_set": [utterance for utterance in utterances if utterance.get("text") != ""]
        }

        res = requests.post(f"{SERVER}:{PORT}/api/podcasts/{slug}/{ep_guid}/utterances/", json=segmentation_dict)
        print(res.status_code)
    except Exception as e:
        failed.append(transcript)
        print(e)
        print("SpaCy failed again", slug, ep_guid, lang )

Special procedure for Swedish because of lots of dashes that Whisper is inserting and is causing issues with spaCy

In [ ]:
import spacy
from spacy.tokens import Span
import copy

for transcript in transcripts:
    slug = transcript["slug"]
    ep_guid = transcript["ep_guid"]
    trans_uuid = transcript["trans_uuid"]

    # Get the text from the API
    data = requests.get(f"{SERVER}:{PORT}/api/transcriptions/{trans_uuid}/")
    transcript = data.json()
    
    lang = transcript["item"]["channel"]["language"][0:2]
    if lang == "sv":
        nlp = spacy.load("sv_core_news_lg")

        # Set the custom attributes for start and end times on Span objects
        Span.set_extension('start_time', default=None, force=True)
        Span.set_extension('end_time', default=None, force=True)

        # load the words with start and end times
        words = copy.deepcopy(transcript['words'])
        # Concatenate the words into a text string
        text = "".join([word["word"] for word in words])
        # Process the text
        doc = nlp(text.strip())
        # load the diarization
        dz = transcript["diarization"]["content"]
        utterances = []
        sentence_list = list(doc.sents)
        idx = 0
        # Iterate over the sentences
        print(sentence_list)
        while idx < len(sentence_list):
            sent = sentence_list[idx]
            # Initialize start and end times
            start_time = words[0]["start"]
            sent_concat = ""
            while sent.text.strip("-– ") != sent_concat.strip("-– "):
                word = words.pop(0)
                sent_concat += word["word"]
                # if sent.text.strip() doesn't end with period, question mark or exclamation mark
                # check if sent_concat matches sent.text.strip() + following sent.text.strip()
                # special rare case when spaCy is not able to split the sentence correctly
                if sent.text.strip()[-1] not in [".", "?", "!"] and idx < len(sentence_list) - 1:
                    if sent.text.strip() + sentence_list[idx+1].text.strip() == sent_concat.strip():
                        idx += 1
                        break
            idx += 1

            end_time = word["end"]
            # Assign start and end times to the sentence
            sent._.start_time = start_time
            sent._.end_time = end_time
            # Append the sentence to the list of utterances
            utterances.append({"text": sent.text.replace("– –", " ").replace("- -", " ").replace("- –", " ").replace("– -", " "), "start": sent._.start_time, "end": sent._.end_time})
            segmentation_dict = {
                "uuid": trans_uuid,
                "name": "spaCy",
                "segmentor": {"name": "spaCy", "version": spacy.__version__},
                "utterance_set": [utterance for utterance in utterances if utterance.get("text") != ""]
        }

        res = requests.post(f"{SERVER}:{PORT}/api/podcasts/{slug}/{ep_guid}/utterances/", json=segmentation_dict)
        print(res.status_code)



In [ ]:
nlp = spacy.load("sv_core_news_lg")

# Set the custom attributes for start and end times on Span objects
Span.set_extension('start_time', default=None, force=True)
Span.set_extension('end_time', default=None, force=True)

# load the words with start and end times
words = copy.deepcopy(transcript['words'])
# Concatenate the words into a text string
text = "".join([word["word"] for word in words])
# Process the text
doc = nlp(text.strip())
# load the diarization
dz = transcript["diarization"]["content"]

In [ ]:
utterances = []
sentence_list = list(doc.sents)
print(sentence_list)
idx = 0
# Iterate over the sentences
while idx < len(sentence_list):
    sent = sentence_list[idx]
    # Initialize start and end times
    start_time = words[0]["start"]
    sent_concat = ""
    while sent.text.strip("-– ") != sent_concat.strip("-– "):
        word = words.pop(0)
        sent_concat += word["word"]
        # if sent.text.strip() doesn't end with period, question mark or exclamation mark
        # check if sent_concat matches sent.text.strip() + following sent.text.strip()
        # special rare case when spaCy is not able to split the sentence correctly
        if sent.text.strip()[-1] not in [".", "?", "!"] and idx < len(sentence_list) - 1:
            if sent.text.strip() + sentence_list[idx+1].text.strip() == sent_concat.strip():
                idx += 1
                break
    idx += 1

    end_time = word["end"]
    # Assign start and end times to the sentence
    sent._.start_time = start_time
    sent._.end_time = end_time
    # Append the sentence to the list of utterances
    utterances.append({"text": sent.text.replace("– –", " ").replace("- -", " ").replace("- –", " ").replace("– -", " "), "start": sent._.start_time, "end": sent._.end_time})
print(utterances)


In [ ]:
word

In [ ]:
utterances

In [ ]:
utest = {"test1": "hello"}
utest

In [ ]:
utest.push("hey", "there")

## FILTER TO NEW SEGMENTATION

In [2]:
import requests

#PORT = 8008
#SERVER = "http://127.0.0.1"

SERVER = "http://192.168.2.239"
PORT = 80

podcasts = requests.get(f"{SERVER}:{PORT}/api/podcasts/")
podcasts = podcasts.json()

Checkworthiness

In [ ]:
import numpy as np
import pandas as pd
import spacy
import requests

for podcast in podcasts:
    slug = podcast['slug']
    for audioitem in podcast['audioitem_set']:
        ep_guid = audioitem['guid']
        for transcription in audioitem['transcription_set']:
            trans_uuid = transcription['uuid']
            for segmentation in transcription['segmentation_set']:
                if segmentation['name'] == "spaCy":
                    seg_uuid = segmentation['uuid']
                    seg = requests.get(f"{SERVER}:{PORT}/api/segmentations/{seg_uuid}/")
                    seg = seg.json()

                    df = pd.DataFrame(seg["utterance_set"])
                    df["score"] = None

                    # get the "ClaimBuster-BBA-(COREF)" score for each utterance where it exists, otherwise use the "ClaimBuster-BBA" score
                    def get_score(classification_set):
                        score = next((float(cl["label"]) for cl in classification_set if cl["agent"] == "ClaimBuster-BBA-(COREF)"), None)
                        if score is None:
                            score = next((float(cl["label"]) for cl in classification_set if cl["agent"] == "ClaimBuster-BBA"), None)
                            if score is None:
                                score = 0
                        return score
                    
                    df["score"] = df["classification_set"].apply(get_score)
                    # set all records that score below the 95th percentile to hidden
                    df.loc[df["score"] < df["score"].quantile(0.95), "visibility"] = 0
                    new_utterances = df.drop(columns=["score"]).to_dict(orient="records")


                    segmentation_dict = {
                        "uuid": trans_uuid,
                        "name": "spaCy-5% most CW",
                        "segmentor": {"name": "spaCy", "version": spacy.__version__},
                        "utterance_set": new_utterances
                    }
                    res = requests.post(f"{SERVER}:{PORT}/api/podcasts/{slug}/{ep_guid}/utterances/", json=segmentation_dict)
                    print(res.status_code)


Add new segmentation for Prolific trial run Transcription/Diarization/Advertising, with given podcast GUIDs

In [ ]:
#guids = [
#    "48723cca-0541-4efd-9f81-affe0046c365", # Ted Cruz, Hunter Biden Bombshells plus Rep. Jim Jordan Joins Us for Deep Dive on Biden Crime Family Part 1
#    "01c17681-f1d1-4b4e-bac8-b000000ec0f8", # Ted Cruz, Deep Dive into Hunter Biden & the Weaponization of the Federal Government: Part 2 with Jim Jordan
#    "0319727e-0b17-4b00-b0ee-02da95c624ac", # Pod Save America, Trump’s CNN Clown Hall
#    "ea7f5719-43e4-462a-bf94-3d6eb8b1b9d2", # Pod Save America, Tucker’s War With Fox
#]

In [ ]:
guids = [
    "a5a6b78c-08f2-488d-9733-4b3cb8419dc6", # 01:15:07 PSA, Trump’s Surprise Q-chella Set
    "b09bc151-d7ce-434a-91a0-2606136e8aed", # 01:06:01 PSA, Democrats Go Off In An Off Year
    "dbcb6c4f-0175-409c-af6d-b0000025ebb3", # 00:42:47 Ted Cruz, Border Chaos: LIVE
    "4245c02d-d92f-464e-bb2a-b00500264713", # 00:47:18 Ted Cruz, Bombshell Durham Report Vindicates Trump
    "5c21d2bd-51fa-46e6-b564-b0060154225d", # 00:40:26 Ted Cruz, FBI Caught Red-Handed
    "645d6c01eff373001114a886", # 00:32:03 New Abnormal, George Santos Is the ‘Inventing Anna’ of Congress
    "6462a31110dbac0011ce3ac0", # 00:57:55 New Abnormal, How Chasten Buttigieg Gets Revenge Against Right Wing Bigots
    "6459895bd2b1090011a103c3", # 00:44:14 New Abnormal, Florida’s House Republicans Are Even Worse Behind the Scenes
    "280f08b0-a3e6-11ed-9b89-6b2526b95d5e", #* 00:18:51 Fox News Rundown, Evening Edition: Durham Report Sharply Criticizes FBI And DOJ
    "28478032-a3e6-11ed-9b89-836b6f8b51fd", #* 00:13:45 Fox News Rundown, Evening Edition: Could The Debt Default Bring The Demise Of The Dollar?
    "28b94096-a3e6-11ed-9b89-0bd8ac5b8052", #* 00:12:20 Fox News Rundown, Evening Edition: Will The Israel-Gaza Ceasefire Last Or Is A Worse Conflict Coming?
    "httpsapispreakercomepisode53923082", #* 00:15:59 Ancient Health Podcast, 190: Man's Best Microbiome Buddy—How Dogs Benefit Our Gut Health
    "httpsapispreakercomepisode53742883", #* 00:23:07 Ancient Health Podcast, 186. Deep Dive: Navigating Anti-Nutrients and How To Prepare Meals for Optimal Digestion
    "4e287b44-4a4c-46e7-a425-4e3d406ca728", #* 00:19:25 ZOE, The surprising health impact of eating too fast
    "f411f4d9-c1f5-44e7-90c6-acfbb7d1abda", # 00:19:21 ZOE, Omega-3 supplements: why you're (probably) wasting your money
    "d4dd0ab7-f6ab-41a1-93b6-72c504e2fec9", # 00:15:48 ZOE, Why eating nuts makes you healthier, according to science
    "645d637162ead30011e344e0", # 00:15:14 The Doctor's Farmacy, How To Stay Asleep And Sleep More Deeply
    "64551437c2f2f20011b7226b", # 00:21:34 The Doctor's Farmacy, Is Hormonal Imbalance The Cause Of Your Resistance To Weight Loss?
    "d2594683-d703-4a33-9f17-b7947684b3d9", # 00:26:40 The Daily, The Day Title 42 Ended
    "89a7f115-9f0d-488f-a44b-1766c38ac6d9", # 00:25:08 The Daily, Biden’s Radical Option to End the Debt Fight
    "ce7e03d8-9279-4fae-954d-412b91253b16", # 00:22:13 The Daily, The U.S. Banned Spyware — and Then Kept Trying to Use It
]


In [3]:
import numpy as np
import pandas as pd
import spacy
import requests

for podcast in podcasts:
    slug = podcast['slug']
    #if slug != pod_slug:
    #    continue
    for audioitem in podcast['audioitem_set']:
        #if audioitem['guid'] not in guids:
        #    continue
        ep_guid = audioitem['guid']
        for transcription in audioitem['transcription_set']:
            trans_uuid = transcription['uuid']
            seg_exists = False
            for segmentation in transcription['segmentation_set']:
                if segmentation['name'] == "Factiverse CW/MO/CS":
                    seg_exists = True
                    break
            if seg_exists:
                continue
            for segmentation in transcription['segmentation_set']:
                if segmentation['name'] == "spaCy":
                    seg_uuid = segmentation['uuid']
                    # retrieve full transcript from API
                    seg = requests.get(f"{SERVER}:{PORT}/api/segmentations/{seg_uuid}/")
                    seg = seg.json()
                    for utterance in seg['utterance_set']:
                        utterance['visibility'] = ["Checkworthiness", "Motivation", "ClaimSpan"]

                    segmentation_dict = {
                        "uuid": trans_uuid,
                        "name": "Factiverse CW/MO/CS",
                        "segmentor": {"name": "spaCy", "version": spacy.__version__},
                        "utterance_set": seg["utterance_set"],
                        "agentsession_set": [],
                    }
                    res = requests.post(f"{SERVER}:{PORT}/api/podcasts/{slug}/{ep_guid}/utterances/", json=segmentation_dict)
                    print(res.status_code)

201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201


In [ ]:
res.text

In [ ]:
import numpy as np
import pandas as pd
import spacy
import requests

for podcast in podcasts:
    slug = podcast['slug']
    for audioitem in podcast['audioitem_set']:
        ep_guid = audioitem['guid']
        for transcription in audioitem['transcription_set']:
            trans_uuid = transcription['uuid']
            for segmentation in transcription['segmentation_set']:
                if segmentation['name'] == "spaCy":
                    seg_uuid = segmentation['uuid']
                    seg = requests.get(f"{SERVER}:{PORT}/api/segmentations/{seg_uuid}/")
                    seg = seg.json()

                    df = pd.DataFrame(seg["utterance_set"])
                    df["score"] = None

                    # get the "ClaimBuster-BBA-(COREF)" score for each utterance where it exists, otherwise use the "ClaimBuster-BBA" score
                    def get_score(classification_set):
                        score = next((float(cl["label"]) for cl in classification_set if cl["agent"] == "ClaimBuster-BBA-(COREF)"), None)
                        if score is None:
                            score = next((float(cl["label"]) for cl in classification_set if cl["agent"] == "ClaimBuster-BBA"), None)
                            if score is None:
                                score = 0
                        return score
                    
                    df["score"] = df["classification_set"].apply(get_score)
                    # set all records that score below the 95th percentile to hidden
                    df.loc[df["score"] < df["score"].quantile(0.95), "visibility"] = 0
                    new_utterances = df.drop(columns=["score"]).to_dict(orient="records")


                    segmentation_dict = {
                        "uuid": trans_uuid,
                        "name": "spaCy-5% most CW",
                        "segmentor": {"name": "spaCy", "version": spacy.__version__},
                        "utterance_set": new_utterances
                    }
                    res = requests.post(f"{SERVER}:{PORT}/api/podcasts/{slug}/{ep_guid}/utterances/", json=segmentation_dict)
                    print(res.status_code)


In [ ]:
# find the 5% of utterances with the highest average Checkworthiness score from both CB & FV
import numpy as np
import pandas as pd
df = pd.DataFrame(seg["utterance_set"])

# get the average Checkworthiness score for each utterance, from the label field of the classification objects
df["score"] = df["classification_set"].apply(lambda x: np.mean([float(cl["label"]) for cl in x if cl["category"] == "Checkworthy" and (cl["agent"] == "ClaimBuster-BBA" or cl["agent"] == "Factiverse")]))
